In [1]:
!pip install ipython==8.12.0

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import os, sys

USE_DRIVE = False
GH_REPO   = "https://github.com/47v0/cs4782-final-project.git"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/gdrive")
    PROJECT_ROOT = "/content/gdrive/MyDrive/2025-2026/Sem 6/CS 4782/final_project/cs4782-final-project"
else:
    PROJECT_ROOT = "/content/cs4782-final-project"

if not os.path.exists(PROJECT_ROOT):
    !git clone {GH_REPO} "{PROJECT_ROOT}"
else:
    # Force-sync to origin/main. This discards uncommitted local edits so the
    # cell can't silently fail on stale Drive working state. Safe because all
    # real changes are committed to GitHub via the local repo.
    !cd "{PROJECT_ROOT}" && git fetch origin && git reset --hard origin/main && git clean -fd

CODE_DIR = f"{PROJECT_ROOT}/code"
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
!ls "{PROJECT_ROOT}"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Updating files: 100% (20/20), done.
HEAD is now at 9fcad6e Merge branch 'main' of github.com:47v0/cs4782-final-project
PROJECT_ROOT: /content/gdrive/MyDrive/2025-2026/Sem 6/CS 4782/final_project/cs4782-final-project
2211.14730v2.pdf  data			  LICENSE   poster     report
code		  final_deliverables.pdf  plan.txt  README.md  results


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- shape constants ---
B = 4         # small batch for prototyping
M = 7         # ETTh1 channels
L = 336       # look-back
T = 96        # forecast horizon

# --- patching ---
P = 16        # patch length
S = 8         # stride

# --- transformer (small-dataset override per Appendix A.1.4) ---
D       = 16     # latent dim
H       = 4      # attention heads
F_DIM   = 128    # feed-forward hidden
LAYERS  = 3
DROPOUT = 0.2

torch.manual_seed(2021)   # paper's seed

# Sanity: number of patches per channel
N = (L - P) // S + 2
print(f"Patches per channel N = (L-P)//S + 2 = ({L}-{P})//{S} + 2 = {N}")

Patches per channel N = (L-P)//S + 2 = (336-16)//8 + 2 = 42


In [6]:
# A fake batch of look-back windows, the shape our DataLoader will yield.
# We use random data with deliberate per-channel offsets/scales so RevIN
# in the next cell has something interesting to normalize.

x = torch.randn(B, M, L) * torch.linspace(1, 7, M).view(1, M, 1)  # different scale per channel
x = x + torch.linspace(-3, 3, M).view(1, M, 1)                     # different offset per channel

print("x.shape:", tuple(x.shape))
print("per-channel mean (sample 0):", x[0].mean(dim=-1).round(decimals=2).tolist())
print("per-channel std  (sample 0):", x[0].std (dim=-1).round(decimals=2).tolist())

x.shape: (4, 7, 336)
per-channel mean (sample 0): [-2.9800000190734863, -1.7799999713897705, -0.9900000095367432, 0.3400000035762787, 1.4299999475479126, 2.4700000286102295, 3.440000057220459]
per-channel std  (sample 0): [1.0499999523162842, 1.9800000190734863, 2.9100000858306885, 3.8399999141693115, 5.150000095367432, 5.570000171661377, 6.900000095367432]


In [7]:
# Per-sample, per-channel normalization across the time axis.
# Save mean & std so we can add them back at the very end.

revin_mean = x.mean(dim=-1, keepdim=True)               # (B, M, 1)
revin_std  = x.std (dim=-1, keepdim=True) + 1e-5        # (B, M, 1)

x_norm = (x - revin_mean) / revin_std                    # (B, M, L)

print("x_norm.shape:", tuple(x_norm.shape))
print("per-channel mean after RevIN (sample 0):",
      x_norm[0].mean(dim=-1).round(decimals=4).tolist())
print("per-channel std  after RevIN (sample 0):",
      x_norm[0].std (dim=-1).round(decimals=4).tolist())

x_norm.shape: (4, 7, 336)
per-channel mean after RevIN (sample 0): [-0.0, 0.0, 0.0, 0.0, -0.0, 0.0, -0.0]
per-channel std  after RevIN (sample 0): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [8]:
# Merge channel dim into batch so each channel is its own "sample" for
# the shared-weights transformer. This is the channel-independence trick.

x_ci = x_norm.reshape(B * M, L)                 # (B*M, L)

print("before reshape:", tuple(x_norm.shape))
print("after  reshape:", tuple(x_ci.shape))
print(f"B*M = {B}*{M} = {B*M}")

before reshape: (4, 7, 336)
after  reshape: (28, 336)
B*M = 4*7 = 28


In [9]:
# 1. Replicate the last value S times to the end of each series.
# 2. Unfold into overlapping length-P windows with stride S.

last_val = x_ci[:, -1:].expand(-1, S)                  # (B*M, S) — copies of last timestep
x_padded = torch.cat([x_ci, last_val], dim=-1)         # (B*M, L+S)

x_patched = x_padded.unfold(dimension=-1, size=P, step=S)  # (B*M, N, P)

print("after replicate-pad:", tuple(x_padded.shape))
print("after unfold:       ", tuple(x_patched.shape))
print(f"sanity: N from formula = (L-P)//S + 2 = {(L-P)//S + 2}")


after replicate-pad: (28, 344)
after unfold:        (28, 42, 16)
sanity: N from formula = (L-P)//S + 2 = 42


In [10]:
patch_proj = nn.Linear(P, D)             # 16 → 16 for ETTh1
x_proj     = patch_proj(x_patched)       # (B*M, N, D)

print("after projection:", tuple(x_proj.shape))
print(f"learnable params: P*D + D = {P}*{D} + {D} = {P*D + D}")


after projection: (28, 42, 16)
learnable params: P*D + D = 16*16 + 16 = 272


In [11]:
# Paper: W_pos in R^{D x N}, learnable additive. Same embedding broadcast
# across the B*M instances so it tags "patch index", not "which sample".

pos_emb = nn.Parameter(torch.randn(1, N, D) * 0.02)   # (1, N, D)
x_pos   = x_proj + pos_emb                             # (B*M, N, D)

print("pos_emb shape:", tuple(pos_emb.shape))
print("after pos emb:", tuple(x_pos.shape))
print(f"learnable params: N*D = {N}*{D} = {N*D}")

pos_emb shape: (1, 42, 16)
after pos emb: (28, 42, 16)
learnable params: N*D = 42*16 = 672


In [12]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model         = D,
    nhead           = H,
    dim_feedforward = F_DIM,
    dropout         = DROPOUT,
    activation      = "gelu",        # paper uses GELU
    batch_first     = True,          # so input is (B, N, D), not (N, B, D)
)
encoder = nn.TransformerEncoder(encoder_layer, num_layers=LAYERS)

x_enc = encoder(x_pos)               # (B*M, N, D)

print("after encoder:", tuple(x_enc.shape))
print(f"encoder params: {sum(p.numel() for p in encoder.parameters()):,}")

after encoder: (28, 42, 16)
encoder params: 16,176


In [13]:
# Per channel-instance: flatten (N, D) tokens into one big vector,
# then map to T forecast steps with a single Linear layer.

flatten = nn.Flatten(start_dim=-2)              # (B*M, N, D) -> (B*M, N*D)
head    = nn.Linear(N * D, T)                   # (N*D=672) -> (T=96)

x_flat  = flatten(x_enc)                        # (B*M, N*D) = (28, 672)
y_hat   = head(x_flat)                          # (B*M, T)   = (28, 96)

print("after flatten:", tuple(x_flat.shape))
print("after head:   ", tuple(y_hat.shape))
print(f"head params: N*D*T + T = {N}*{D}*{T} + {T} = {N*D*T + T:,}")

after flatten: (28, 672)
after head:    (28, 96)
head params: N*D*T + T = 42*16*96 + 96 = 64,608


In [14]:
# Un-merge channel from batch
y_pred = y_hat.reshape(B, M, T)                  # (B*M, T) -> (B, M, T)

# Undo instance normalization: multiply by std, add mean.
# revin_std and revin_mean are (B, M, 1) -> broadcast across the T axis.
y_pred = y_pred * revin_std + revin_mean        # (B, M, T)

print("final prediction shape:", tuple(y_pred.shape))
print("\nper-channel mean of pred (sample 0):",
      y_pred[0].mean(dim=-1).round(decimals=2).tolist())
print("per-channel std  of pred (sample 0):",
      y_pred[0].std (dim=-1).round(decimals=2).tolist())
print("\noriginal input means (sample 0):",
      x[0].mean(dim=-1).round(decimals=2).tolist())

final prediction shape: (4, 7, 96)

per-channel mean of pred (sample 0): [-2.9000000953674316, -1.75, -1.25, 0.4699999988079071, 1.5299999713897705, 2.119999885559082, 3.2200000286102295]
per-channel std  of pred (sample 0): [0.5699999928474426, 1.159999966621399, 1.6399999856948853, 2.2699999809265137, 3.0999999046325684, 3.2200000286102295, 4.0]

original input means (sample 0): [-2.9800000190734863, -1.7799999713897705, -0.9900000095367432, 0.3400000035762787, 1.4299999475479126, 2.4700000286102295, 3.440000057220459]


In [15]:
from model import PatchTST

model = PatchTST(
    seq_len=L, pred_len=T, patch_len=P, stride=S,
    n_features=M, d_model=D, n_heads=H, n_layers=LAYERS,
    d_ff=F_DIM, dropout=DROPOUT,
)

y_full = model(x)

print("model output shape:", tuple(y_full.shape))
print("expected:          ", (B, M, T))
print(f"total params: {sum(p.numel() for p in model.parameters()):,}")

model output shape: (4, 7, 96)
expected:           (4, 7, 96)
total params: 7,024
